#### GBDT for Recommendation – Using Python, load a sample recommendation dataset (for example, the MovieLens 100K dataset). Preprocess it to create a user-item interaction table with features (include user genre preferences, item popularity, etc.). Train a LightGBM classifier to predict whether a user will interact with a given item. Provide the code to train the model, and evaluate it using AUC or NDCG.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from tqdm import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, ndcg_score

In [2]:
# Load the data
users = pd.read_csv('ml-100k/u.user', sep='|', header=None, 
                   names=['user_id', 'age', 'gender', 'occupation', 'zip_code'])
items = pd.read_csv('ml-100k/u.item', sep='|', header=None, encoding='latin-1',
                    names=['item_id', 'title', 'release_date', 'video_release_date', 
                          'IMDb_URL'] + [f'genre_{i}' for i in range(19)])
ratings = pd.read_csv('ml-100k/u.data', sep='\t', header=None,
                     names=['user_id', 'item_id', 'rating', 'timestamp'])

In [3]:
# Convert ratings to binary interactions (1 for ratings >= 4, 0 otherwise)
ratings['interaction'] = (ratings['rating'] >= 4).astype(int)

# Calculate item popularity features
item_popularity = ratings.groupby('item_id').agg({
    'user_id': 'count',  # number of ratings
    'rating': ['mean', 'std'],  # rating statistics
    'interaction': 'mean'  # positive interaction rate
}).fillna(0)

item_popularity.columns = ['total_ratings', 'avg_rating', 'rating_std', 'positive_rate']
item_popularity = item_popularity.reset_index()

In [4]:
# Create user features
# Calculate user genre preferences based on their ratings
user_genre_prefs = pd.DataFrame(index=users['user_id'].unique())
genre_cols = [col for col in items.columns if 'genre_' in col]

# Merge ratings with items to get genre information for each rating
ratings_with_genres = ratings.merge(items[['item_id'] + genre_cols], on='item_id')

# Calculate average rating for each genre per user
for genre in genre_cols:
    genre_ratings = ratings_with_genres[ratings_with_genres[genre] == 1].groupby('user_id')['rating'].agg(['mean', 'count'])
    user_genre_prefs[f'{genre}_avg'] = genre_ratings['mean']
    user_genre_prefs[f'{genre}_count'] = genre_ratings['count']

user_genre_prefs[[f'{col}_count' for col in genre_cols]] = user_genre_prefs[[f'{col}_count' for col in genre_cols]].fillna(0)

# Add user statistics
user_stats = ratings.groupby('user_id').agg({
    'rating': ['count', 'mean', 'std'],
    'interaction': 'mean'
}).fillna(0)
user_stats.columns = ['total_ratings', 'avg_rating', 'rating_std', 'positive_rate']
user_stats = user_stats.reset_index()

# One-hot encode user occupation
users_encoded = pd.get_dummies(users, columns=['occupation', 'gender'])

In [7]:
# Process release date to get movie age
items['release_date'] = pd.to_datetime(items['release_date'], format='%d-%b-%Y', errors='coerce')
items['movie_age'] = (pd.Timestamp('1998-04-23') - items['release_date']).dt.days

# Combine all features
def create_features(user_id, item_id):
    # User features
    user_features = users_encoded[users_encoded['user_id'] == user_id].iloc[0].to_dict()
    user_stats_features = user_stats[user_stats['user_id'] == user_id].iloc[0].to_dict()
    user_genre_features = user_genre_prefs.loc[user_id].to_dict()
    
    # Item features
    item_features = items[items['item_id'] == item_id].iloc[0][genre_cols + ['movie_age']].to_dict()
    item_pop_features = item_popularity[item_popularity['item_id'] == item_id].iloc[0].to_dict()
    
    # Combine all features
    features = {**user_features, **user_stats_features, **user_genre_features, 
               **item_features, **item_pop_features}
    return features

# Create training dataset
interactions = ratings[['user_id', 'item_id', 'interaction']].copy()
feature_list = []

# Create features for each interaction
for _, row in tqdm(interactions.sample(10000, random_state=42).iterrows()):
    features = create_features(row['user_id'], row['item_id'])
    feature_list.append(features)

# Convert to DataFrame
feature_df = pd.DataFrame(feature_list)

10000it [00:35, 285.27it/s]



In [8]:
feature_df[['user_id', 'item_id']] = feature_df[['user_id', 'item_id']].astype('int64')
feature_df['zip_code'] = feature_df['zip_code'].astype('category')

In [9]:
combined_data = feature_df.merge(interactions, on=['user_id', 'item_id'])

In [10]:
# Prepare final dataset
X = combined_data.drop(columns=['interaction'])
y = combined_data['interaction']

In [15]:
X.columns

Index(['user_id', 'age', 'zip_code', 'occupation_administrator',
       'occupation_artist', 'occupation_doctor', 'occupation_educator',
       'occupation_engineer', 'occupation_entertainment',
       'occupation_executive', 'occupation_healthcare', 'occupation_homemaker',
       'occupation_lawyer', 'occupation_librarian', 'occupation_marketing',
       'occupation_none', 'occupation_other', 'occupation_programmer',
       'occupation_retired', 'occupation_salesman', 'occupation_scientist',
       'occupation_student', 'occupation_technician', 'occupation_writer',
       'gender_F', 'gender_M', 'total_ratings', 'avg_rating', 'rating_std',
       'positive_rate', 'genre_0_avg', 'genre_0_count', 'genre_1_avg',
       'genre_1_count', 'genre_2_avg', 'genre_2_count', 'genre_3_avg',
       'genre_3_count', 'genre_4_avg', 'genre_4_count', 'genre_5_avg',
       'genre_5_count', 'genre_6_avg', 'genre_6_count', 'genre_7_avg',
       'genre_7_count', 'genre_8_avg', 'genre_8_count', 'genre_9_av

In [ ]:
# Calculate NDCG for each user
def calculate_ndcg_at_k(group, k=10):
    true_relevance = group['interaction'].values
    predicted_scores = group['predicted_score'].values
    return ndcg_score([true_relevance], [predicted_scores], k=min(k, len(true_relevance)))

# Remove user_id and item_id from training features
feature_cols = [col for col in X.columns if col not in ['user_id', 'item_id']]
categorical_features = [col for col in X.columns if col.startswith('occupation_')] + ['gender_F', 'gender_M'] + ['zip_code'] + genre_cols

# Initialize and train the model
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9
}

# Five-fold cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = []
ndcg_scores = []

# Show feature importance
feature_importance = pd.DataFrame()

for fold, (train_idx, test_idx) in enumerate(kf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    feature_cols = [col for col in X_train.columns if col not in ['user_id', 'item_id']]

    train_data = lgb.Dataset(X_train[feature_cols], label=y_train, categorical_feature=categorical_features)
    valid_data = lgb.Dataset(X_test[feature_cols], label=y_test, reference=train_data, categorical_feature=categorical_features)

    model = lgb.train(
        params,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=100,
        callbacks=[lgb.early_stopping(10)]
    )

    feature_importance = pd.concat([feature_importance, 
                                    pd.DataFrame({'features': feature_cols, 'importance':model.feature_importance(), 'fold':[fold+1]*len(feature_cols)})], ignore_index=True)

    y_pred = model.predict(X_test[feature_cols])
    auc = roc_auc_score(y_test, y_pred)
    auc_scores.append(auc)

    # Prepare data for NDCG calculation
    test_predictions = pd.DataFrame({
        'user_id': X_test['user_id'],
        'item_id': X_test['item_id'],
        'interaction': y_test,
        'predicted_score': y_pred
    })
    user_counts = test_predictions.groupby('user_id')['user_id'].count()
    ndcg = test_predictions[test_predictions.user_id.isin(user_counts[user_counts > 1].index)].groupby('user_id').apply(calculate_ndcg_at_k, k=10).mean()
    ndcg_scores.append(ndcg)
    print(f"Fold {fold+1}: AUC={auc:.4f}, NDCG@10={ndcg:.4f}")

print(f"\nMean AUC: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print(f"Mean NDCG@10: {np.mean(ndcg_scores):.4f} ± {np.std(ndcg_scores):.4f}")

In [181]:
print("\nTop 10 most important features:")
print(feature_importance.groupby('features')['importance'].mean().sort_values(ascending=False).head(10))


Top 10 most important features:
features
positive_rate    121.4
avg_rating        48.0
zip_code          43.8
genre_8_avg       41.6
genre_1_avg       40.8
genre_16_avg      30.4
genre_14_avg      29.0
genre_5_avg       23.6
genre_6_avg       20.8
total_ratings     19.4
Name: importance, dtype: float64


#### Using PyTorch, define an embedding layer for users and for items (each of size 50) and then compute a predicted rating via a dot product.

In [13]:
import torch
import torch.nn as nn

class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=50):
        super().__init__()
        # Create embedding layers for users and items
        self.user_embeddings = nn.Embedding(num_embeddings=n_users, embedding_dim=embedding_dim)
        self.item_embeddings = nn.Embedding(num_embeddings=n_items, embedding_dim=embedding_dim)
        
        # Initialize embeddings with small random values
        nn.init.normal_(self.user_embeddings.weight, std=0.01)
        nn.init.normal_(self.item_embeddings.weight, std=0.01)
        
    def forward(self, user_ids, item_ids):
        # Get embeddings for users and items
        user_embeds = self.user_embeddings(user_ids)  # shape: (batch_size, embedding_dim)
        item_embeds = self.item_embeddings(item_ids)  # shape: (batch_size, embedding_dim)
        
        # Compute dot product between user and item embeddings
        predictions = torch.sum(user_embeds * item_embeds, dim=1)  # element-wise multiply and sum
        return torch.sigmoid(predictions)  # scale to [0,1] for binary prediction

In [20]:
# Initialize model
n_users = users['user_id'].nunique()
n_items = items['item_id'].nunique()
model = MatrixFactorization(n_users, n_items, embedding_dim=50)

# Example forward pass
batch_size = 32
user_ids = torch.randint(0, n_users, (batch_size,))  # random user IDs
item_ids = torch.randint(0, n_items, (batch_size,))  # random item IDs

# Get predictions
predictions = model(user_ids, item_ids)
print(f"Shape of predictions: {predictions.shape}")
print(f"Sample predictions:\n{predictions[:5]}")

# To see the actual embeddings
print("\nShape of user embeddings:", model.user_embeddings.weight.shape)
print("Shape of item embeddings:", model.item_embeddings.weight.shape)

# Example: Get embedding for a specific user and item
user_id = torch.tensor([0])
item_id = torch.tensor([0])
user_embedding = model.user_embeddings(user_id)
item_embedding = model.item_embeddings(item_id)
print(f"\nSample user embedding (first 5 dimensions): {user_embedding[0, :5]}")
print(f"Sample item embedding (first 5 dimensions): {item_embedding[0, :5]}")

Shape of predictions: torch.Size([32])
Sample predictions:
tensor([0.5001, 0.5000, 0.5002, 0.5001, 0.5002], grad_fn=<SliceBackward0>)

Shape of user embeddings: torch.Size([943, 50])
Shape of item embeddings: torch.Size([1682, 50])

Sample user embedding (first 5 dimensions): tensor([-0.0136, -0.0013,  0.0085,  0.0034, -0.0122], grad_fn=<SliceBackward0>)
Sample item embedding (first 5 dimensions): tensor([ 0.0076, -0.0079,  0.0076,  0.0181,  0.0022], grad_fn=<SliceBackward0>)


In [ ]:
from torch.utils.data import Dataset, DataLoader

class MovieLensDataset(Dataset):
    def __init__(self, interactions_df):
        self.users = torch.tensor(interactions_df['user_id'].values, dtype=torch.long)
        self.items = torch.tensor(interactions_df['item_id'].values, dtype=torch.long)
        self.labels = torch.tensor(interactions_df['interaction'].values, dtype=torch.float)
        
    def __len__(self):
        return len(self.users)
    
    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

# Create train/test split
train_data, test_data = train_test_split(interactions, test_size=0.2, random_state=42)

# Create datasets and dataloaders
train_dataset = MovieLensDataset(train_data)
test_dataset = MovieLensDataset(test_data)

batch_size = 1024
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Training setup
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
num_epochs = 5

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_users, batch_items, batch_labels in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
        # Forward pass
        predictions = model(batch_users, batch_items)
        loss = criterion(predictions, batch_labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    # Validation
    model.eval()
    val_preds = []
    val_labels = []
    
    with torch.no_grad():
        for batch_users, batch_items, batch_labels in test_loader:
            predictions = model(batch_users, batch_items)
            val_preds.extend(predictions.numpy())
            val_labels.extend(batch_labels.numpy())
    
    # Calculate metrics
    val_preds = np.array(val_preds)
    val_labels = np.array(val_labels)
    auc = roc_auc_score(val_labels, val_preds)
    
    print(f'Epoch {epoch+1}:')
    print(f'Average Training Loss: {total_loss/len(train_loader):.4f}')
    print(f'Validation AUC: {auc:.4f}')

#### Write a function to compute precision@K and recall@K for a single user’s recommendations. The function takes a list of recommended item IDs and a set of that user’s true relevant item IDs as inputs.

In [26]:
def precision_recall_at_k(recommended_items, relevant_items, k=10):
    """
    Compute precision@K and recall@K for a single user.
    Args:
        recommended_items (list): List of recommended item IDs (ordered by predicted relevance).
        relevant_items (set): Set of true relevant item IDs for the user.
        k (int): Number of top recommendations to consider.
    Returns:
        precision (float): Precision@K
        recall (float): Recall@K
    """
    assert k > 0
    assert len(recommended_items) >= k
    recommended_at_k = recommended_items[:k]
    hits = set(recommended_at_k) & set(relevant_items)
    precision = len(hits) / k
    recall = len(hits) / len(relevant_items) if relevant_items else 0.0
    return precision, recall

# Example usage:
recommended = [10, 20, 30, 40, 50]
relevant = {20, 30, 60}
p, r = precision_recall_at_k(recommended, relevant, k=3)
print(f"Precision@3: {p:.2f}, Recall@3: {r:.2f}")# Prepare training data

Precision@3: 0.67, Recall@3: 0.67
